<a href="https://colab.research.google.com/github/igrosh30/EmployeeAttendanceSystem/blob/master/DigitalWifiModulations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from statistics import mode
from scipy.special import erfc# calculate BER

N=  -90 #same val of noise in dBm,

# 1- Experimental Measurments

Here the goal is to have 3 measurments points of the recieved power that need to lie between:


1.   High:   [-35 ; -40]dBm;
2.   Medium: [-65 ; -70]dBm;
3.   Low: [-80;...]dBm;

> **Create a DataFrame**: for every point have 10 values of PHY velocity and the mode; 5 values for the recieved power and the mean;Wi-Fi chanel bandwith and the corresponding digital modulation.


**Note:** the digital modulation is determined with the help of table:
[Digital Wi-Fi modulation Table](https://mcsindex.net/)

## Measured values

In [4]:
power_list = {
    "1":[-40,-35,-35,-38,-40,-36,-39,-40,-37,-36],
    "2":[-66,-69,-70,-67,-65,-66,-69,-70,-65,-66],
    "3":[-82,-82,-81,-82,-83,-82,-84,-85,-86,-89]
}
vel_list = {
    "1":[96,96,86,96,96,96,96,96,96,96],
    "2": [65,39,57,57,57,57,57,43,52,72],
    "3":[6,19,13,52,26,39,26,39,19,6]
}
bandwith = [80,80,20]#in MHz
modulacao = ['QPSK','BPSK','BPSK']

In [5]:
power = []
vel = []
for index in power_list:
  power.append(np.mean(power_list[index]))
  vel.append(mode(vel_list[index]))

print(vel)
print(power)


[96, 57, 6]
[np.float64(-37.6), np.float64(-67.3), np.float64(-83.6)]


In [6]:
df = pd.DataFrame({
    "Potência Média (dBm)": power,
    "Moda Velocidade PHY (Mbit/s)": vel,
    "Largura de Banda (MHz)": bandwith,
    "Modulação Digital": modulacao
    },
    index=[1,2,3])
df.index.name = "Ponto"
df

,Potência Média (dBm),Moda Velocidade PHY (Mbit/s),Largura de Banda (MHz),Modulação Digital
Ponto,,,,
1,-37.6,96,80,QPSK
2,-67.3,57,80,BPSK
3,-83.6,6,20,BPSK


# 2 - Bit Error Probability

To calculate BER for each modulation we need to know the value for $\frac{Eb}{N0} = \frac{B}{Rb}*\frac{S}{N} $


*   Energy per bit will depend on the power and the ratio of bits that we are sending: $Eb=\frac{power}{time}$
*   Noise spectral density, depends on the noise and the corresponding bandwith: $No=\frac{noise}{bandwith}$




In [7]:
def dBToLinear(dB):
  return 10**(dB/10)

# Função BER (BPSK e QPSK têm exatamente a mesma fórmula)
def ber_bpsk_qpsk(EbN0_lin):
    return 0.5 * erfc(np.sqrt(EbN0_lin))

In [8]:
# Cálculos
SNR_dB = df["Potência Média (dBm)"] - N
SNR_linear = SNR_dB.apply(dBToLinear)
bandwith_Hz = df["Largura de Banda (MHz)"] * 1e6
Rb_bps = df["Moda Velocidade PHY (Mbit/s)"] * 1e6

df["Eb/N0 (linear)"] = bandwith_Hz * SNR_linear / Rb_bps
df["Eb/N0(dB)"] = 10 * np.log10(df["Eb/N0 (linear)"])

df["BER"] = df["Eb/N0 (linear)"].apply(ber_bpsk_qpsk)

df


,Potência Média (dBm),Moda Velocidade PHY (Mbit/s),Largura de Banda (MHz),Modulação Digital,Eb/N0 (linear),Eb/N0(dB),BER
Ponto,,,,,,,
1,-37.6,96,80,QPSK,144816.735729,51.608188,0.000000e+00
2,-67.3,57,80,BPSK,261.345563,24.172151,5.495721e-116
3,-83.6,6,20,BPSK,14.550528,11.628787,3.434977e-08


## Grpahs

# 3- Analysis of the BER and Coding Rate